# 03 — Parseo de Criterios de Elegibilidad

Separación de la columna `eligibility_criteria` en listas de criterios de **inclusión** y **exclusión** individuales.

**Estrategia en cascada:**
1. Regex agresivo sobre headers `Inclusion Criteria:` / `Exclusion Criteria:` y variantes
2. Parseo de ítems por bullets, numeración o saltos de línea
3. Textos sin headers reconocibles → `llm_pending` (procesamiento LLM futuro)
4. Flag de calidad: `clean` / `partial` / `llm_pending`

**Input:** `data/clinical_trials.parquet`  
**Output:** `data/03_eligibility_parsed.parquet`

## 3.1 Configuration

In [4]:
from pathlib import Path
import pandas as pd
import sys
import re
sys.path.append("../utils")
from download_parquet import download_dataset

DOWNLOAD_DATA  = False
PARQUET_INPUT  = Path('..') / 'data' / 'clinical_trials.parquet'
PARQUET_OUTPUT = Path('..') / 'data' / '03_eligibility_parsed.parquet'
MIN_ITEM_CHARS = 10  # descarta ítems más cortos que esto

## 3.2 Load Dataset

In [5]:
if DOWNLOAD_DATA:
    download_dataset()
else:
    print('DOWNLOAD_DATA=False — usando parquet local.')

df_studies = pd.read_parquet(PARQUET_INPUT)
print(f'Estudios cargados: {len(df_studies):,}')
df_studies[['nct_id', 'eligibility_criteria']].head(3)

DOWNLOAD_DATA=False — usando parquet local.
Estudios cargados: 83,962


,nct_id,eligibility_criteria
0,NCT00728442,"Inclusion Criteria:\n\n* Non metastatic, inclu..."
1,NCT01859442,Inclusion Criteria:\n\n* diagnosis of T3N+ rec...
2,NCT05741242,Patients must satisfy the following criteria t...


## 3.3 Diagnóstico del campo `eligibility_criteria`

In [6]:
texts = df_studies['eligibility_criteria'].dropna()
total = len(texts)

# Nota: pandas usa PyArrow/RE2 internamente — usar caracteres literales, sin \• ni \ en clases
has_incl   = texts.str.contains(r'(?i)inclusion criteria|inclusion:', regex=True)
has_excl   = texts.str.contains(r'(?i)exclusion criteria|exclusion:', regex=True)
has_both   = has_incl & has_excl
has_bullet = texts.str.contains(r'(?m)^\s*[*\-•]', regex=True)
has_num    = texts.str.contains(r'(?m)^\s*\d+[.)]', regex=True)
no_headers = ~(has_incl | has_excl)

print(f'Total no-nulos:               {total:>7,}')
print(f'Tienen Inclusion header:      {has_incl.sum():>7,}  ({has_incl.mean():.1%})')
print(f'Tienen Exclusion header:      {has_excl.sum():>7,}  ({has_excl.mean():.1%})')
print(f'Tienen ambos:                 {has_both.sum():>7,}  ({has_both.mean():.1%})')
print(f'Bullets (*/-/•):              {has_bullet.sum():>7,}  ({has_bullet.mean():.1%})')
print(f'Numeración:                   {has_num.sum():>7,}  ({has_num.mean():.1%})')
print(f'Sin ningún header (→ llm):    {no_headers.sum():>7,}  ({no_headers.mean():.1%})')

Total no-nulos:                83,958
Tienen Inclusion header:       78,563  (93.6%)
Tienen Exclusion header:       76,722  (91.4%)
Tienen ambos:                  76,460  (91.1%)
Bullets (*/-/•):               66,919  (79.7%)
Numeración:                    23,293  (27.7%)
Sin ningún header (→ llm):      5,133  (6.1%)


## 3.4 Funciones de parseo

In [7]:
# Regex que matchea cualquier variante de header de inclusión o exclusión
_HEADER_RE = re.compile(
    r'(?i)'
    r'(?P<incl>(?:key |main |principal )?inclusion(?:\s+criteria)?(?:\s+criterion)?\s*:?)'
    r'|'
    r'(?P<excl>(?:key |main |principal )?exclusion(?:\s+criteria)?(?:\s+criterion)?\s*:?)'
)


def split_by_headers(text: str) -> tuple[str | None, str | None, str]:
    """Parte el texto en secciones de inclusión y exclusión usando regex de headers.

    Retorna (inclusion_raw, exclusion_raw, method).
    method puede ser 'both_headers', 'incl_only', 'excl_only', o 'no_header'.
    """
    matches = list(_HEADER_RE.finditer(text))
    if not matches:
        return None, None, 'no_header'

    segments: list[tuple[str, int, int]] = []
    for i, m in enumerate(matches):
        tipo = 'incl' if m.group('incl') else 'excl'
        content_start = m.end()
        content_end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        segments.append((tipo, content_start, content_end))

    # Texto antes del primer header → inclusión implícita (preamble)
    preamble = text[:matches[0].start()].strip()

    incl_parts: list[str] = []
    excl_parts: list[str] = []

    if preamble:
        incl_parts.append(preamble)

    for tipo, cs, ce in segments:
        chunk = text[cs:ce].strip()
        if tipo == 'incl':
            incl_parts.append(chunk)
        else:
            excl_parts.append(chunk)

    incl_raw = '\n'.join(incl_parts).strip() or None
    excl_raw = '\n'.join(excl_parts).strip() or None

    if incl_raw and excl_raw:
        method = 'both_headers'
    elif incl_raw:
        method = 'incl_only'
    else:
        method = 'excl_only'

    return incl_raw, excl_raw, method

In [8]:
# Estos usan re de Python (no RE2), así que los literales son seguros en ambos
_BULLET_RE = re.compile(r'(?m)^\s*[*\-•·]\s+')
_NUMBER_RE = re.compile(r'(?m)^\s*\d+[.)]\s+')


def parse_items(section_text: str) -> list[str]:
    """Extrae ítems individuales de una sección de texto."""
    if not section_text:
        return []

    # Intento 1: split por bullets
    if _BULLET_RE.search(section_text):
        raw_items = _BULLET_RE.split(section_text)
    # Intento 2: split por numeración
    elif _NUMBER_RE.search(section_text):
        raw_items = _NUMBER_RE.split(section_text)
    # Intento 3: párrafos separados por doble salto
    elif '\n\n' in section_text:
        raw_items = section_text.split('\n\n')
    # Fallback: cada línea es un ítem
    else:
        raw_items = section_text.split('\n')

    return [
        item.strip()
        for item in raw_items
        if item.strip() and len(item.strip()) >= MIN_ITEM_CHARS
    ]

In [9]:
def assign_quality(
    method: str,
    inclusion: list[str],
    exclusion: list[str],
) -> str:
    """Asigna parse_status según la completitud del parseo."""
    if method == 'no_header':
        return 'llm_pending'
    if method == 'both_headers' and len(inclusion) >= 2 and len(exclusion) >= 2:
        return 'clean'
    return 'partial'

In [10]:
def parse_eligibility(text: str) -> dict:
    """Orquesta el parseo completo de un texto de elegibilidad."""
    if not isinstance(text, str) or not text.strip():
        return {
            'inclusion_criteria': [],
            'exclusion_criteria': [],
            'parse_status':       'llm_pending',
            '_incl_raw':          None,
            '_excl_raw':          None,
            '_original_len':      0,
        }

    incl_raw, excl_raw, method = split_by_headers(text)
    inclusion = parse_items(incl_raw) if incl_raw else []
    exclusion = parse_items(excl_raw) if excl_raw else []
    status    = assign_quality(method, inclusion, exclusion)

    return {
        'inclusion_criteria': inclusion,
        'exclusion_criteria': exclusion,
        'parse_status':       status,
        '_incl_raw':          incl_raw,
        '_excl_raw':          excl_raw,
        '_original_len':      len(text),
    }

## 3.5 Pipeline

In [11]:
parsed_rows = df_studies['eligibility_criteria'].apply(parse_eligibility)
df_parsed = pd.DataFrame(parsed_rows.tolist())
df_parsed.insert(0, 'nct_id', df_studies['nct_id'].values)

print(f'Filas parseadas: {len(df_parsed):,}')
print(df_parsed['parse_status'].value_counts())

Filas parseadas: 83,962
parse_status
clean          64848
partial        14122
llm_pending     4992
Name: count, dtype: int64


## 3.6 Control de calidad (interno)

Métricas para detectar pérdida de información durante el split. Estas columnas **no se persisten** en el parquet final.

In [14]:
def compute_char_retention(row: pd.Series) -> float | None:
    """Ratio de caracteres recuperados respecto al texto original."""
    if row['_original_len'] == 0:
        return None
    incl_len = len(row['_incl_raw']) if pd.notna(row['_incl_raw']) else 0
    excl_len = len(row['_excl_raw']) if pd.notna(row['_excl_raw']) else 0
    return (incl_len + excl_len) / row['_original_len']


df_parsed['_char_retention'] = df_parsed.apply(compute_char_retention, axis=1)
df_parsed['_n_items_total']  = (
    df_parsed['inclusion_criteria'].apply(len) +
    df_parsed['exclusion_criteria'].apply(len)
)
df_parsed['_has_preamble'] = df_parsed['_incl_raw'].apply(
    lambda x:  pd.notna(x) and bool(x) and not bool(_HEADER_RE.match(x.lstrip()))
)

qa_summary = df_parsed.groupby('parse_status').agg(
    n                 = ('nct_id', 'count'),
    retention_median  = ('_char_retention', 'median'),
    retention_p10     = ('_char_retention', lambda x: x.quantile(0.10)),
    pct_low_retention = ('_char_retention', lambda x: (x < 0.70).mean()),
    items_median      = ('_n_items_total', 'median'),
)
print(qa_summary.to_string())
print(f"\nEstudios con retención < 70%: {(df_parsed['_char_retention'] < 0.70).sum():,}")
print(f"Estudios con preamble detectado: {df_parsed['_has_preamble'].sum():,}")

                  n  retention_median  retention_p10  pct_low_retention  items_median
parse_status                                                                         
clean         64848          0.972050       0.908333           0.000370          16.0
llm_pending    4992          0.000000       0.000000           0.999199           0.0
partial       14122          0.902439       0.741803           0.068121           4.0

Estudios con retención < 70%: 5,974
Estudios con preamble detectado: 78,955


In [15]:
# Inspección manual de casos con baja retención (excluye llm_pending)
low_ret = df_parsed[
    (df_parsed['_char_retention'] < 0.70) &
    (df_parsed['parse_status'] != 'llm_pending')
].head(3)

for _, row in low_ret.iterrows():
    original = df_studies.loc[df_studies['nct_id'] == row['nct_id'], 'eligibility_criteria'].values[0]
    print(f"NCT: {row['nct_id']} | status: {row['parse_status']} | retention: {row['_char_retention']:.2f}")
    print(f"Original ({len(original)} chars):\n{original[:300]}")
    print(f"Incl raw: {repr(str(row['_incl_raw'])[:200])}")
    print(f"Excl raw: {repr(str(row['_excl_raw'])[:200])}")
    print('---')

NCT: NCT01310972 | status: partial | retention: 0.66
Original (131 chars):
Inclusion Criteria:

* Patients with Adjuvant Colon Cancer

Exclusion Criteria:

* Patients with diagnosis other than Colon Cancer.
Incl raw: '* Patients with Adjuvant Colon Cancer'
Excl raw: '* Patients with diagnosis other than Colon Cancer.'
---
NCT: NCT05498402 | status: partial | retention: 0.59
Original (108 chars):
Inclusion Criteria:

* Being a registered EMT, or paramedic

Exclusion Criteria:

* Member of the study team
Incl raw: '* Being a registered EMT, or paramedic'
Excl raw: '* Member of the study team'
---
NCT: NCT02874001 | status: partial | retention: 0.70
Original (146 chars):
Inclusion Criteria:

* cancer
* having a caregiver support time
* Aged 18 years and over

Exclusion Criteria:

* without known psychiatric history
Incl raw: '* cancer\n* having a caregiver support time\n* Aged 18 years and over'
Excl raw: '* without known psychiatric history'
---


## 3.7 Output

In [17]:
OUTPUT_COLS = ['nct_id', 'inclusion_criteria', 'exclusion_criteria', 'parse_status']

df_output = df_parsed[OUTPUT_COLS].copy()
df_output['n_inclusion'] = df_output['inclusion_criteria'].apply(len)
df_output['n_exclusion'] = df_output['exclusion_criteria'].apply(len)

df_output.to_parquet(PARQUET_OUTPUT, index=False)

print(f'Guardado: {PARQUET_OUTPUT}')
print(f'Filas: {len(df_output):,}')
print('\nDistribución parse_status:')
print(df_output['parse_status'].value_counts())
df_output.head(3)

Guardado: ..\data\03_eligibility_parsed.parquet
Filas: 83,962

Distribución parse_status:
parse_status
clean          64848
partial        14122
llm_pending     4992
Name: count, dtype: int64


,nct_id,inclusion_criteria,exclusion_criteria,parse_status,n_inclusion,n_exclusion
0,NCT00728442,"[Non metastatic, including invasive and in sit...","[Breast disease without cancer, Metastatic bre...",clean,2,5
1,NCT01859442,"[diagnosis of T3N+ rectal cancer, above age of...",[unable to conduct a cardiopulmonary exercise ...,clean,3,2
2,NCT05741242,[Patients must satisfy the following criteria ...,[History of serious adverse reaction to vaccin...,clean,8,7
